In [ ]:
import pandas as pd
import numpy as np
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path
from typing import Optional
import matplotlib.pyplot as plt
import polars as pl
import psutil
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# 1. Coinmetrics

In [ ]:
# read data
df = pd.read_csv("../data/Coin Metrics/coinmetrics_btc.csv")
df["time"] = pd.to_datetime(df["time"])
df.head()

In [ ]:
# test the commit
start = "2018-01-01"
end = "2025-12-31"
df = df[(df["time"] >= start) & (df["time"] <= end)].copy()
# sort and set the index
df = df.sort_values("time").set_index("time")

In [ ]:
# check missing date
all_days = pd.date_range(start, end, freq="D")
missing_days = all_days.difference(df.index)
len(missing_days)

In [ ]:
# missing rate
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print("missing rate Top 20（%）：")
print(missing_pct.head(20))

key_cols = [
    "PriceUSD", "CapMrktCurUSD", "HashRate", "TxCnt",
    "CapMVRVCur", "SplyCur", "FeeTotNtv",
    "FlowInExUSD", "FlowOutExUSD",
    "volume_reported_spot_usd_1d"
]
key_cols = [c for c in key_cols if c in df.columns]

print("\nkey row describe：")
print(df[key_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)

In [ ]:
# Basic integrity checks
print("Rows:", len(df))
print("Time range:", df.index.min(), "→", df.index.max())

print(df.index.is_monotonic_increasing)
print(df.index.is_unique)

print("\nMissing values (top 10 columns):")
print(df.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
# 6) create metrics

px = df["PriceUSD"]

df["ret_1d"] = px.pct_change()
df["logret_1d"] = np.log(px).diff()

# annualized vol
df["vol_30d"] = df["logret_1d"].rolling(30).std() * np.sqrt(365)

# 200 DMA
df["ma_200"] = px.rolling(200).mean()
df["price_vs_ma200"] = px / df["ma_200"] - 1

# Drawdown
rolling_max = px.cummax()
df["drawdown"] = px / rolling_max - 1


In [ ]:
df.columns

In [ ]:
df.head()

# 2. Coinmetrics EDA

In [ ]:
metrics = ["PriceUSD", "CapMrktCurUSD", "HashRate"]
df[metrics].describe()

In [ ]:
# Price Trend
plt.figure(figsize=(12, 4))
plt.plot(df.index, df["PriceUSD"])
plt.title("BTC Price — 2018-01-01 to 2025-12-31")
plt.xlabel("Date")
plt.ylabel("PriceUSD")
plt.tight_layout()
plt.show()

In [ ]:
cols = ["PriceUSD", "CapMrktCurUSD", "HashRate"]
scaled = StandardScaler().fit_transform(df[cols])

plt.figure(figsize=(12,4))
plt.plot(df.index, scaled)
plt.legend(cols)
plt.title("Standardized BTC Metrics")
plt.tight_layout()
plt.savefig("plots/Standardized BTC Metrics.png", dpi=300)
plt.show()

In [ ]:
# BTC Price (Log Scale) with 200-Day Moving Average
plt.figure(figsize=(12, 6))
plt.plot(df.index, df["PriceUSD"], label="BTC Price (USD)")
plt.plot(df.index, df["ma_200"], label="200-Day Moving Average", linestyle="--")

plt.yscale("log")

plt.title("BTC Price (Log Scale) with 200-Day Moving Average")
plt.xlabel("Date")
plt.ylabel("Price (USD, Log Scale)")

plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Daily Profit Distribution
rets = df["ret_1d"].dropna()
plt.figure(figsize=(10, 4))
plt.hist(rets, bins=200)
plt.title("Daily Returns Histogram")
plt.xlabel("Daily Return")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
print("\nWorst 10 Days (ret_1d):")
print(rets.nsmallest(10))

print("\nBest 10 Days (ret_1d):")
print(rets.nlargest(10))

In [ ]:
# 30D Rolling Annualized Volatility
plt.figure(figsize=(12, 4))
plt.plot(df.index, df["vol_30d"])
plt.title("30D Rolling Annualized Volatility")
plt.xlabel("Date")
plt.ylabel("Annualized Volatility")
plt.tight_layout()
plt.show()

In [ ]:
# Drawdown Plot
plt.figure(figsize=(12, 4))
plt.plot(df.index, df["drawdown"])
plt.title("Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.tight_layout()
plt.show()

In [ ]:
#Correlation Heatmap
correlation_cols = ["PriceUSD", "CapMrktCurUSD", "HashRate", "TxCnt"]
corr_df = df[correlation_cols].apply(pd.to_numeric, errors="coerce")

# corr
corr = corr_df.corr()

# heatmap
plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap="viridis", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
corr_cols = [
    "PriceUSD", "CapMrktCurUSD", "HashRate", "TxCnt", "TxTfrCnt",
    "CapMVRVCur", "SplyCur", "AdrActCnt", "AdrBalCnt",
    "FeeTotNtv", "FlowInExUSD", "FlowOutExUSD",
    "volume_reported_spot_usd_1d"
]
corr_cols = [c for c in corr_cols if c in df.columns]

corr = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="viridis", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("plots/correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Build accumulation common metrics
# net inflow >0 means net outflow ，<0 means net inflow
df["net_flow_usd"] = df["FlowOutExUSD"] - df["FlowInExUSD"]

# PnL in next 30 days
h = 30
df["fwd_ret_30d"] = df["PriceUSD"].shift(-h) / df["PriceUSD"] - 1

# Forward vol in 30D log return
logret = np.log(df["PriceUSD"]).diff()
df["fwd_vol_30d"] = logret.shift(-1).rolling(h).std() * np.sqrt(365)

# Max Drawdown in next 30D
# Make the lowest price in the forward window, then convert to max drawdown
future_min_price = df["PriceUSD"].shift(-1).rolling(h).min()
df["fwd_min_dd_30d"] = future_min_price / df["PriceUSD"] - 1 

# Remove missing data in the last 30D
eda = df.dropna(subset=["fwd_ret_30d", "fwd_vol_30d", "fwd_min_dd_30d"]).copy()


In [ ]:
#One dimensional bin analysis 
#Group into 5 binnings by MVRV
eda["mvrv_q5"] = pd.qcut(eda["CapMVRVCur"], 5, labels=[1,2,3,4,5])
mvrv_table = eda.groupby("mvrv_q5").agg(
    n=("PriceUSD", "size"),
    avg_fwd_ret_30d=("fwd_ret_30d", "mean"),
    med_fwd_ret_30d=("fwd_ret_30d", "median"),
    avg_fwd_vol_30d=("fwd_vol_30d", "mean"),
    prob_dd_lt_20pct=("fwd_min_dd_30d", lambda x: (x < -0.20).mean()),
    prob_dd_lt_30pct=("fwd_min_dd_30d", lambda x: (x < -0.30).mean()),
).reset_index()

print("\n===  MVRV（CapMVRVCur）5 Buckets Grouping Result ===")
print(mvrv_table)

plt.figure(figsize=(10,4))
plt.plot(mvrv_table["mvrv_q5"], mvrv_table["avg_fwd_ret_30d"], marker="o")
plt.title("MVRV Quintiles vs Future 30-Day Average Returns")
plt.xlabel("MVRV Quintile (1=Undervalued, 5=Overvalued)")
plt.ylabel("Average fwd_ret_30d")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Based on net_flow_usd group into 5
eda["netflow_q5"] = pd.qcut(eda["net_flow_usd"], 5, labels=[1,2,3,4,5])
netflow_table = eda.groupby("netflow_q5").agg(
    n=("PriceUSD", "size"),
    avg_fwd_ret_30d=("fwd_ret_30d", "mean"),
    med_fwd_ret_30d=("fwd_ret_30d", "median"),
    avg_fwd_vol_30d=("fwd_vol_30d", "mean"),
    prob_dd_lt_20pct=("fwd_min_dd_30d", lambda x: (x < -0.20).mean()),
    prob_dd_lt_30pct=("fwd_min_dd_30d", lambda x: (x < -0.30).mean()),
).reset_index()

print("\n=== Net Flow (net_flow_usd) Quintile Binning Results ===")
print(netflow_table)

plt.figure(figsize=(10,4))
plt.plot(netflow_table["netflow_q5"], netflow_table["prob_dd_lt_20pct"], marker="o")
plt.title("Net Flow Quintiles vs Future 30-Day Drawdown <-20% Probability")
plt.xlabel("Net Flow Quintile (1=Strong Inflow, 5=Strong Outflow)")
plt.ylabel("P(fwd_min_dd_30d < -20%)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#  AdrActCnt 5 binning
eda["adract_q5"] = pd.qcut(eda["AdrActCnt"], 5, labels=[1,2,3,4,5])
adr_table = eda.groupby("adract_q5").agg(
    n=("PriceUSD", "size"),
    avg_fwd_ret_30d=("fwd_ret_30d", "mean"),
    med_fwd_ret_30d=("fwd_ret_30d", "median"),
    avg_fwd_vol_30d=("fwd_vol_30d", "mean"),
    prob_dd_lt_20pct=("fwd_min_dd_30d", lambda x: (x < -0.20).mean()),
    prob_dd_lt_30pct=("fwd_min_dd_30d", lambda x: (x < -0.30).mean()),
).reset_index()

print("\n=== （AdrActCnt）Quintile Binning Results ===")
print(adr_table)

plt.figure(figsize=(10,4))
plt.plot(adr_table["adract_q5"], adr_table["avg_fwd_vol_30d"], marker="o")
plt.title("AdrActCnt Quintiles vs Future 30-Day Average Volatility")
plt.xlabel("AdrActCnt Quintile (1=Low Activity, 5=High Activity)）")
plt.ylabel("Average fwd_vol_30d（Annualized）")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# # Forward-looking performance conditioning analysis
# # 1 Build forward-looking metrics
# df["net_flow_usd"] = df["FlowOutExUSD"] - df["FlowInExUSD"]

# h = 30
# #
# # Forward 30-day return
# df["fwd_ret_30d"] = df["PriceUSD"].shift(-h) / df["PriceUSD"] - 1

# # Forward 30-day volatility (annualized)
# logret = np.log(df["PriceUSD"]).diff()
# df["fwd_vol_30d"] = logret.shift(-1).rolling(h).std() * np.sqrt(365)

# # Forward 30-day max drawdown
# future_min_price = df["PriceUSD"].shift(-1).rolling(h).min()
# df["fwd_min_dd_30d"] = future_min_price / df["PriceUSD"] - 1

# # Remove NaN
# eda = df.dropna(subset=["fwd_ret_30d", 
#                         "fwd_vol_30d", 
#                         "fwd_min_dd_30d"]).copy()

# # 2 Quintile grouping
# # MVRV
# eda["mvrv_q5"] = pd.qcut(eda["CapMVRVCur"], 5, labels=[1,2,3,4,5])
# mvrv_table = eda.groupby("mvrv_q5").agg(
#     avg_fwd_ret_30d=("fwd_ret_30d", "mean"),
#     avg_fwd_vol_30d=("fwd_vol_30d", "mean"),
#     prob_dd_lt_20pct=("fwd_min_dd_30d", lambda x: (x < -0.20).mean()),
# ).reset_index()

# # Net Flow
# eda["netflow_q5"] = pd.qcut(eda["net_flow_usd"], 5, labels=[1,2,3,4,5])
# netflow_table = eda.groupby("netflow_q5").agg(
#     avg_fwd_ret_30d=("fwd_ret_30d", "mean"),
#     avg_fwd_vol_30d=("fwd_vol_30d", "mean"),
#     prob_dd_lt_20pct=("fwd_min_dd_30d", lambda x: (x < -0.20).mean()),
# ).reset_index()

# # Active Address
# eda["adract_q5"] = pd.qcut(eda["AdrActCnt"], 5, labels=[1,2,3,4,5])
# adr_table = eda.groupby("adract_q5").agg(
#     avg_fwd_ret_30d=("fwd_ret_30d", "mean"),
#     avg_fwd_vol_30d=("fwd_vol_30d", "mean"),
#     prob_dd_lt_20pct=("fwd_min_dd_30d", lambda x: (x < -0.20).mean()),
# ).reset_index()

# 3 visualization
fig, axes = plt.subplots(1, 3, figsize=(18,5))

# MVRV
axes[0].plot(mvrv_table["mvrv_q5"], 
             mvrv_table["avg_fwd_ret_30d"], marker="o", label="Avg Return")
axes[0].plot(mvrv_table["mvrv_q5"], 
             mvrv_table["avg_fwd_vol_30d"], marker="s", label="Avg Volatility")
axes[0].plot(mvrv_table["mvrv_q5"], 
             mvrv_table["prob_dd_lt_20pct"], marker="^", label="P(Drawdown<-20%)")

axes[0].set_title("MVRV Quintiles")
axes[0].set_xlabel("Quintile")
axes[0].grid(True)

# Net Flow 
axes[1].plot(netflow_table["netflow_q5"], 
             netflow_table["avg_fwd_ret_30d"], marker="o")
axes[1].plot(netflow_table["netflow_q5"], 
             netflow_table["avg_fwd_vol_30d"], marker="s")
axes[1].plot(netflow_table["netflow_q5"], 
             netflow_table["prob_dd_lt_20pct"], marker="^")

axes[1].set_title("Net Flow Quintiles")
axes[1].set_xlabel("Quintile")
axes[1].grid(True)

# Active Address
axes[2].plot(adr_table["adract_q5"], 
             adr_table["avg_fwd_ret_30d"], marker="o")
axes[2].plot(adr_table["adract_q5"], 
             adr_table["avg_fwd_vol_30d"], marker="s")
axes[2].plot(adr_table["adract_q5"], 
             adr_table["prob_dd_lt_20pct"], marker="^")

axes[2].set_title("Active Address Quintiles")
axes[2].set_xlabel("Quintile")
axes[2].grid(True)


# Unified legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)

plt.tight_layout(rect=[0, 0, 1, 0.92])

plt.savefig("plots/Forward_30D_Factor_Comparison.png",
            dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# 2D Binning: MVRV × Netflow
pivot = eda.pivot_table(
    index="mvrv_q5",
    columns="netflow_q5",
    values="fwd_ret_30d",
    aggfunc="mean"
)

plt.figure(figsize=(9,6))
sns.heatmap(pivot, annot=True, cmap="viridis", fmt=".3f")

plt.title("2D Binning: MVRV Quintile × Netflow Quintile\nAverage 30-Day Forward Return")
plt.xlabel("Netflow Quintile (1 = Net Inflow ↑ Selling Pressure, 5 = Net Outflow ↑ Accumulation)")
plt.ylabel("MVRV Quintile (1 = Undervalued, 5 = Overvalued)")

plt.tight_layout()
plt.savefig("plots/2D Binning: MVRV Quintile × Netflow Quintile\nAverage 30-Day Forward Return.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#  How much of the drawdown is recovered in the following 7 days?
eda['recovery_7d'] = eda['PriceUSD'].shift(-7) / eda['PriceUSD'] - 1

# Forward Return Probability based on Drawdown Depth
drawdown_buckets = pd.cut(eda['drawdown'], bins=[-1, -0.3, -0.15, -0.05, 0], labels=['Crash', 'Correction', 'Pullback', 'Stable'])

recovery_stats = eda.groupby(drawdown_buckets)['recovery_7d'].agg(['mean', 'std', 'count'])
print("BTC Recovery Velocity by Drawdown Regime:")
print(recovery_stats)

# Plotting different BTC Drawdown regimes
plt.figure(figsize=(10, 5))
sns.boxplot(x=drawdown_buckets, y=eda['recovery_7d'], palette="RdYlGn")
plt.title("7-Day Forward Returns by BTC Drawdown Depth")
plt.axhline(0, color='black', linestyle='--')
plt.show()

# 3. Polymarket data loading

In [ ]:
#read polymarket data
import sys
import os
import polars as pl
#from pathlib import path

sys.path.append(os.path.abspath('..'))
# Import the loader from the template provided by the project
from eda_starter_template import load_polymarket_data, POLYMARKET_DIR

# Load the dictionary of DataFrames
poly_data_dict = load_polymarket_data(POLYMARKET_DIR)

# Extract individual DataFrames for easier use
if poly_data_dict:
    df_markets = poly_data_dict.get("markets")
    df_odds = poly_data_dict.get("odds")
    df_summary = poly_data_dict.get("summary")
    
    print("Data loaded successfully!")
else:
    print("Data directory not found. Check if the 'data/Polymarket' folder exists.")

In [ ]:
df_markets.head()

In [ ]:
tokens_path = POLYMARKET_DIR / "finance_politics_tokens.parquet"
event_path = POLYMARKET_DIR / "finance_politics_event_stats.parquet"
trades_path = POLYMARKET_DIR / "finance_politics_trades.parquet"

df_tokens = (
    pl.scan_parquet(tokens_path)
    .collect()
)
df_event = (
    pl.scan_parquet(event_path)
    .collect()
)

if trades_path.exists():
    trades_df = pl.scan_parquet(trades_path).collect()
    
    # Fix timestamp corruption
    for col in trades_df.columns:
        if any(x in col.lower() for x in ["timestamp", "trade", "created_at", "end_date"]):
            if trades_df[col].dtype == pl.Datetime or trades_df[col].dtype == pl.Date:
                if not trades_df[col].is_empty() and trades_df[col].max() < datetime(2020, 1, 1):
                    trades_df = trades_df.with_columns((pl.col(col).cast(pl.Int64) * 1000).cast(pl.Datetime))
                    
            # Enforce 2020+ constraint (replace placeholders/zeros with null)
            if trades_df[col].dtype == pl.Datetime or trades_df[col].dtype == pl.Date:
                    trades_df = trades_df.with_columns(
                        pl.when(pl.col(col) < datetime(2020, 1, 1))
                        .then(None)
                        .otherwise(pl.col(col))
                        .alias(col)
                    )
                
    print(f"Loaded {len(trades_df)} trades records.")

In [ ]:
df_markets
# Filter for rows where both are True
both_true = df_markets.filter(
    (pl.col("active") == True) & (pl.col("closed") == True)
)

print(f"Total Markets: {len(df_markets)}")
print(f"Markets with (active=True AND closed=True): {len(both_true)}")

# 3.1 Markets

In [ ]:
df_markets.head()

#find unique category
df_markets['category'].unique()
search_list = ["bitcoin", "btc","crypto","blockchain",'ETH']
# filter crypto related event, and general political event
crypto_markets = df_markets.filter(
    (pl.col("category").is_in([ "Crypto", "Business",'Politics'])) |
    (pl.col("question").str.contains_any(search_list, ascii_case_insensitive=True))
)

crypto_markets['category'].unique()

#check aggregated newly created daily market
daily_pm_features = (
    crypto_markets
    .filter(pl.col("created_at").is_not_null())
    .group_by(pl.col("created_at").dt.date().alias("date"))
    .agg([
        pl.count("market_id").alias("pm_new_market_count"),
        pl.col("volume").sum().alias("pm_daily_volume"),
        # average days the market remain active
        ((pl.col("end_date") - pl.col("created_at")).dt.total_days()).mean().alias("avg_market_duration")
    ])
    .sort("date")
)

# Convert to pandas 
pm_features_pd = daily_pm_features.to_pandas()
pm_features_pd['date'] = pd.to_datetime(pm_features_pd['date'])

# 

In [ ]:
# combine markets with btc price data
btc_daily = eda.reset_index().rename(columns={'time':'date'})
btc_daily['date'] = btc_daily['date'].dt.normalize()

#merge the daily aggregated polymarket features with BTC price and vol
merged_markets_btc = btc_daily.merge(pm_features_pd,on='date',how='inner')

#correlation matrix
cols_to_corr = ['PriceUSD', 'vol_30d', 'pm_new_market_count', 'pm_daily_volume']
print("Correlation between BTC and Polymarket Activity:")
corr = merged_markets_btc[cols_to_corr].corr()
print(corr)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="viridis", fmt=".2f")
plt.title("Correlation Heatmap for daily event of polymarket vs BTC price")
plt.tight_layout()
plt.show()

In [ ]:
#investigate any lead lag relationship
merged_markets_btc['pm_count_lag1'] = merged_markets_btc['pm_new_market_count'].shift(1)

# Calculate correlation with Price Change (Returns) instead of absolute Price
merged_markets_btc['ret_1d'] = merged_markets_btc['PriceUSD'].pct_change()

lead_corr = merged_markets_btc[['ret_1d', 'pm_count_lag1']].corr()
print("Whether yesterday's PM activity predict today's BTC return?")
print(lead_corr)

In [ ]:
# Check event slug level
#1 event cluster shows election tops the chart (as expected)
event_clusters = (
    df_markets
    .group_by("event_slug")
    .agg([
        pl.col("volume").sum().alias("total_event_volume"),
        pl.count("market_id").alias("sub_market_count"),
        pl.col("active").mean().alias("%_active"),
        pl.col("created_at").min().alias("start_date"),
        pl.col("end_date").max().alias("end_date")
    ])
    .sort("total_event_volume", descending=True)
    .head(10) 
)

print("Top 10 Macro Event Clusters by Volume:")
print(event_clusters)

# Visualization of Event Clusters
plt.figure(figsize=(10, 5))
sns.barplot(data=event_clusters.to_pandas(), x="total_event_volume", y="event_slug", palette="viridis")
plt.title("Volume by Event Cluster")
plt.xlabel("Total USD Volume")
plt.show()

#2 plot BTC price
plt.figure(figsize=(14, 7))

# Plot BTC Price
plt.plot(btc_daily['date'], btc_daily['PriceUSD'], color='black', alpha=0.3, label='BTC Price')
plt.ylabel("Bitcoin Price (USD)")

#3 overlay the cluster, check btc movement during big macro event
colors = ['#ff9999','#66b3ff','#99ff99','#ffcc99', '#c2c2f0']

for i, row in enumerate(event_clusters.to_dicts()):
    # Convert to pandas timestamps for plotting
    start = pd.to_datetime(row['start_date'])
    end = pd.to_datetime(row['end_date'])
    
    # Shade the area where this Macro event was active
    plt.axvspan(start, end, alpha=0.2, color=colors[i % len(colors)], 
                label=f"Macro Event: {row['event_slug']} (${row['total_event_volume']/1e6:.1f}M)")

plt.title("BTC Price Action during 'Macro' Event Windows", fontsize=16)
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
#Check volatility change when event approaches the end

major_slug = event_clusters['event_slug'][6] 

magnet_markets = df_markets.filter(pl.col("event_slug") == major_slug)

# 2. Calculate "Days to Resolution" for each day in history
# We join the BTC daily dates with the event's end date
resolution_date = magnet_markets['end_date'].max()

btc_pressure = btc_daily.copy()
btc_pressure['days_to_res'] = (pd.to_datetime(resolution_date) - pd.to_datetime(btc_pressure['date'])).dt.days

# Filter to only show the 90 days leading up to the resolution
btc_pressure = btc_pressure[(btc_pressure['days_to_res'] >= 0) & (btc_pressure['days_to_res'] <= 90)]

# 3. Plot Volatility vs Countdown
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.set_xlabel('Days to Resolution (Counting Down)')
ax1.set_ylabel('BTC 30d Volatility', color='tab:red')
ax1.plot(btc_pressure['days_to_res'], btc_pressure['vol_30d'], color='tab:red', linewidth=2)
ax1.invert_xaxis() # Countdown from 90 to 0

ax2 = ax1.twinx()
ax2.set_ylabel('BTC Price (USD)', color='tab:blue')
ax2.plot(btc_pressure['days_to_res'], btc_pressure['PriceUSD'], color='tab:blue', linestyle='--')

plt.title(f"BTC Volatility Movement: BTC Behavior Leading to {major_slug} Resolution", fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

# 3.2 Tokens

In [ ]:
df_tokens

In [ ]:
df_tokens.select("outcome").unique()

# 3.3 Event

In [ ]:
df_event.head()

In [ ]:
df_event.describe()

In [ ]:
# df_event 转成 daily event activity
# --- 1) Ensure df_event is pandas + datetime types (force UTC tz-aware) ---
df_event = pd.DataFrame(df_event.to_dict(as_series=False))

df_event["first_market_start"] = pd.to_datetime(
    df_event["first_market_start"], utc=True, errors="coerce")
df_event["last_market_end"] = pd.to_datetime(
    df_event["last_market_end"], utc=True, errors="coerce")

# --- 2) Create daily date range based on BTC df (force UTC tz-aware daily index) ---
daily_index = pd.DatetimeIndex(df.index)

# make sure BTC index is UTC tz-aware
if daily_index.tz is None:
    daily_index = daily_index.tz_localize("UTC")
else:
    daily_index = daily_index.tz_convert("UTC")

daily_index = daily_index.normalize()  # daily dates in UTC

# --- 3) Build daily event activity signals ---
active_event_count = pd.Series(0, index=daily_index, dtype="int64")
active_volume_sum  = pd.Series(0.0, index=daily_index, dtype="float64")
active_market_sum  = pd.Series(0.0, index=daily_index, dtype="float64")

for _, row in df_event.dropna(subset=["first_market_start", "last_market_end"]).iterrows():
    start = row["first_market_start"].normalize()
    end   = row["last_market_end"].normalize()

    # clip to BTC date range (now tz-aware vs tz-aware, safe to compare)
    if end < daily_index.min() or start > daily_index.max():
        continue

    start = max(start, daily_index.min())
    end   = min(end, daily_index.max())

    mask = (daily_index >= start) & (daily_index <= end)
    active_event_count.loc[mask] += 1
    active_volume_sum.loc[mask]  += float(row["total_volume"]) if pd.notna(row["total_volume"]) else 0.0
    active_market_sum.loc[mask]  += float(row["market_count"]) if pd.notna(row["market_count"]) else 0.0

df_event_daily = pd.DataFrame({
    "active_event_count": active_event_count,
    "active_volume_sum": active_volume_sum,
    "active_market_sum": active_market_sum
})

# log scale for volume to reduce extreme skew
df_event_daily["log_active_volume_sum"] = np.log1p(df_event_daily["active_volume_sum"])

# --- 4) Merge with BTC daily features (force BTC daily index to match daily_index) ---
btc_daily = df.copy()

btc_daily.index = pd.DatetimeIndex(btc_daily.index)
if btc_daily.index.tz is None:
    btc_daily.index = btc_daily.index.tz_localize("UTC")
else:
    btc_daily.index = btc_daily.index.tz_convert("UTC")

btc_daily.index = btc_daily.index.normalize()

df_merged = btc_daily.join(df_event_daily, how="left").fillna(0)

df_merged.head()

In [ ]:
#duration
df_event["first_market_start"] = pd.to_datetime(df_event["first_market_start"])
df_event["last_market_end"] = pd.to_datetime(df_event["last_market_end"])

# 计算 duration
df_event["duration_days"] = (
    df_event["last_market_end"] - df_event["first_market_start"]
).dt.days


In [ ]:
#event distribution of volume
plt.figure(figsize=(8,4))
plt.hist(np.log1p(df_event["total_volume"]), bins=60)
plt.title("Distribution of Event Total Volume (log1p)")
plt.xlabel("log1p(total_volume)")
plt.ylabel("Count")
plt.grid(True)
plt.tight_layout()
plt.savefig("plots/event_volume_distribution.png", dpi=300, bbox_inches="tight")
plt.show()
#There is a very tall bar on the left, which means many total_volume values are very small (even close to 0); these events are almost pure noise when you measure “activity intensity”: either nobody is trading, or data quality/market activity is too low.
#The main mass sits in the middle, but there is a long tail on the right,  means a small number of events have extremely large traded volume  This will cause the “high-quantile group” in a quintile split using qcut to be almost entirely dominated by these mega events, while the differences among Q1–Q4 may become quite small.

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df_event["market_count"], bins=50)
plt.title("Distribution of Market Count per Event")
plt.xlabel("market_count")
plt.ylabel("Count")
plt.grid(True)
plt.tight_layout()
plt.savefig("plots/Distribution of Market Count per Event.png", dpi=300, bbox_inches="tight")
plt.show()
# 图中，一个巨大尖峰在最左边（大概率是 1 或 2），右侧只有很少量事件拥有更多 market
# 这会带来一个非常关键的偏差：total_volume 天然会随着 market_count 增加而增加，因为 market 多 = 可下注的合约多 = 成交更容易被“摊大”。
# 也就是说：现在的 total_volume 可能不仅是“事件重要”，而是“事件被拆成了很多市场”。

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df_event["duration_days"], bins=60)
plt.title("Distribution of Event Duration (Days)")
plt.xlabel("duration_days")
plt.ylabel("Count")
plt.grid(True)
plt.tight_layout()
plt.savefig("plots/Distribution of Event Duration (Days).png", dpi=300, bbox_inches="tight")
plt.show()
# duration 图也呈现“尖峰 + 长尾”的形状：说明很多 event 生命周期差不多（比如 30~60 天这种），但也有少数 event 非常长（你表里 1222 天那种）。
# 同样是热门事件，开的越久，累积成交量越大，所以 total_volume 也天然和 duration 强相关。

In [ ]:
top_events = df_event.sort_values("total_volume", ascending=False).head(10)
top_events[["event_slug","market_count","total_volume","duration_days"]]
# top10 基本是：美国大选相关（winner、popular vote、inauguration），地方选举（NYC mayor），Fed 决议（Oct/Dec）等
# 这说明event universe 里“成交量最大的那批”是宏观政治/政策事件，它们确实有可能跟 BTC 风险偏好、波动相关。
# 但同时也能看到：这些事件往往 market_count 更大、duration 更长，所以它们的 total_volume 更容易被抬高。

In [ ]:
fig = plt.figure(figsize=(12, 6))
ax1 = fig.add_subplot(2, 1, 1)
ax2 = fig.add_subplot(2, 1, 2, sharex=ax1)

ax1.plot(df_merged.index, df_merged["PriceUSD"])
ax1.set_title("BTC Price (USD)")

ax2.plot(df_merged.index, df_merged["log_active_volume_sum"])
ax2.set_title("Polymarket Event Activity (log(1 + active_volume_sum))")

plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(12, 6))
ax1 = fig.add_subplot(2, 1, 1)
ax2 = fig.add_subplot(2, 1, 2, sharex=ax1)

ax1.plot(df_merged.index, df_merged["vol_30d"])
ax1.set_title("BTC 30D Annualized Volatility")

ax2.plot(df_merged.index, df_merged["log_active_volume_sum"])
ax2.set_title("Polymarket Event Activity (log(1 + active_volume_sum))")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 3.5))
plt.plot(df_merged.index, df_merged["active_event_count"])
plt.title("Number of Active Polymarket Events per Day")
plt.tight_layout()
plt.show()

In [ ]:
# future
df_merged["fwd_ret_30d"] = (
    df_merged["PriceUSD"].shift(-30) /
    df_merged["PriceUSD"] - 1
)
df_merged["fwd_vol_30d"] = (
    df_merged["vol_30d"]
    .shift(-1)
    .rolling(30)
    .mean()
)
df_merged["fwd_drawdown_30d"] = (
    df_merged["drawdown"]
    .shift(-1)
    .rolling(30)
    .min()
)
df_merged["dd_lt_20pct"] = (
    df_merged["fwd_drawdown_30d"] < -0.20
).astype(int)

In [ ]:
df_merged.columns

In [ ]:
event_table = (
    df_merged
    .dropna(subset=[
        "event_quintile",
        "fwd_ret_30d",
        "fwd_vol_30d",
        "fwd_drawdown_30d"
    ])
    .groupby("event_quintile")
    .agg(
        avg_fwd_ret_30d=("fwd_ret_30d", "mean"),
        avg_fwd_vol_30d=("fwd_vol_30d", "mean"),
        prob_dd_lt_20pct=("dd_lt_20pct", "mean") 
    )
    .reset_index()
)

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))

ax.plot(event_table["event_quintile"],
        event_table["avg_fwd_ret_30d"],
        marker="o", label="Return (30d)")

ax.plot(event_table["event_quintile"],
        event_table["avg_fwd_vol_30d"],
        marker="s", label="Volatility (30d)")

ax.plot(event_table["event_quintile"],
        event_table["prob_dd_lt_20pct"],
        marker="^", label="P(Drawdown < -20%)")

ax.set_title("Event Activity Quintiles – 30d Forward Metrics")
ax.set_xlabel("Quintile (1=Low, 5=High)")
ax.grid(True)
ax.legend()

plt.tight_layout()
plt.show()

# 3.4 Odds History

In [ ]:
df_odds = df_odds.with_columns(
    pl.col("timestamp").dt.date().alias("date")
)

df_odds.head()

In [ ]:
df_tokens_simple = (
    df_tokens
    .with_columns(pl.col("outcome").str.to_lowercase().alias("outcome_lower"))
    .filter(
        pl.col("outcome_lower").is_in(["yes", "no", "up", "down"])
    )
)

In [ ]:
df_odds_simple = df_odds.join(
    df_tokens_simple.select(["market_id", "token_id", "outcome"]),
    on=["market_id", "token_id"],
    how="inner"
)

df_odds_simple.head()

In [ ]:
df_daily = (
    df_odds_simple
    .sort(["market_id", "date", "timestamp"])
    .group_by(["market_id", "date"])
    .agg([
        pl.col("price").last().alias("p_close"),
        pl.count().alias("update_count")
    ])
    .sort(["market_id", "date"])
)

df_daily = df_daily.with_columns(
    (pl.col("p_close") - pl.col("p_close").shift(1))
    .over("market_id")
    .alias("p_diff_1d")
)

df_daily = df_daily.with_columns(
    pl.col("p_diff_1d")
    .rolling_std(7)
    .over("market_id")
    .alias("p_vol_7d")
)

In [ ]:
df_daily.describe()

In [ ]:
daily_index = (
    df_daily
    .group_by("date")
    .agg([
        pl.col("p_vol_7d").mean().alias("risk_index"),
        pl.col("p_diff_1d").mean().alias("prob_shift_index"),
        pl.col("update_count").mean().alias("attention_index"),
        pl.count().alias("num_markets")
    ])
    .sort("date")
)

In [ ]:
daily_pd = pd.DataFrame(daily_index.to_dict(as_series=False))

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(daily_pd["date"], daily_pd["risk_index"])
plt.title("Polymarket Risk Index (Mean 7D Prob Vol)")
plt.xlabel("Date")
plt.ylabel("Risk Index")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(daily_pd["date"], daily_pd["attention_index"])
plt.title("Polymarket Attention Index")
plt.xlabel("Date")
plt.ylabel("Avg Update Count")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
btc_pd = df.reset_index()[["time", "PriceUSD"]]
daily_pd["date"] = pd.to_datetime(daily_pd["date"]).dt.normalize()
btc_pd["time"] = pd.to_datetime(btc_pd["time"]).dt.normalize()

merged = daily_pd.merge(
    btc_pd,
    left_on="date",
    right_on="time",
    how="inner"
)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12,5))

ax1.plot(merged["date"], merged["PriceUSD"], color="blue")
ax1.set_ylabel("BTC Price", color="blue")

ax2 = ax1.twinx()
ax2.plot(merged["date"], merged["risk_index"], color="red", alpha=0.6)
ax2.set_ylabel("Risk Index", color="red")

plt.title("BTC Price vs Polymarket Risk Index")
plt.tight_layout()
plt.show()

In [ ]:
#分箱分析
merged["risk_q5"] = pd.qcut(merged["risk_index"], 5, labels=[1,2,3,4,5])

risk_table = merged.groupby("risk_q5").agg(
    avg_btc_price=("PriceUSD","mean")
)

print(risk_table)

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(risk_table.index, risk_table["avg_btc_price"], marker="o")
plt.title("Risk Quintile vs BTC Avg Price")
plt.xlabel("Risk Quintile")
plt.ylabel("Avg BTC Price")
plt.grid(True)
plt.savefig("plots/Risk_Quintile_vs_BTC_Avg_Price.png",
            dpi=300, bbox_inches="tight")
plt.show()

# 3.5 Trades

In [ ]:
trades_df.head()

In [ ]:
# Identify the top 15 traders by total volume (USD)
TopTraders = (
    trades_df
    .with_columns((pl.col("price") * pl.col("size")).alias("usd_value"))
    .group_by("taker_address")
    .agg(pl.col("usd_value").sum().alias("total_spent"))
    .sort("total_spent", descending=True)
    .head(15)
)

print("Top TopTraders on Polymarket:")
print(TopTraders)

In [ ]:
# # Calculate buy pressure per hour

# buy_sell_pressure = (
#     trades_df
#     .sort("timestamp") 
#     .with_columns(
#         #BUY is positive, everything else (SELL) is negative
#         pl.when(pl.col("side") == "BUY").then(1).otherwise(-1).alias("direction")
#     )
#     .with_columns(
#         # USDvalue
#         (pl.col("direction") * pl.col("size") * pl.col("price")).alias("net_flow")
#     )
#     #Group into 1-hour buckets
#     .group_by_dynamic("timestamp", every="1h")
#     .agg([
#         # Aggregate the net_flow to get the actual "Buy Pressure"
#         pl.col("net_flow").sum().alias("hourly_net_buy_pressure"),
#         pl.col("size").sum().alias("hourly_volume"),
#         pl.count("trade_id").alias("trade_count")
#     ])
# )

# print(buy_sell_pressure.head())

In [ ]:
# Identify concentrated bets - certain address responsible for most volume of bets
address_concentration = (
    trades_df
    .group_by("taker_address")
    .agg(pl.count().alias("trade_count"))
    .sort("trade_count", descending=True)
)


In [ ]:
address_concentration.head(20)

In [ ]:
#Find big trades
bigtrade_threshold = 100000  # $100k+ single trade

big_bets = (
    trades_df
    .select(["market_id", "timestamp", "price", "size", "side"])
    .with_columns((pl.col("price") * pl.col("size")).alias("usd_value"))
    .filter(pl.col("usd_value") > bigtrade_threshold)
)
big_bets.head(20)


In [ ]:
big_markets = big_bets.head(20).join(
    df_markets, 
    on="market_id", 
    how="inner"
)
big_markets.sort("usd_value", descending=True).head(10)



In [ ]:
# 1. Aggregate big trader Volume vs Total Volume per market
whale_stats = (
    trades_df
    .with_columns((pl.col("price") * pl.col("size")).alias("usd_value"))
    .group_by("market_id")
    .agg([
        pl.col("usd_value").sum().alias("total_vol"),
        # Sum of trades specifically from big_bets logic
        pl.col("usd_value").filter(pl.col("usd_value") > 10000).sum().alias("whale_vol"),
        pl.col("taker_address").n_unique().alias("unique_traders")
    ])
    .with_columns([
        (pl.col("whale_vol") / pl.col("total_vol")).alias("whale_dominance")
    ])
    # Filter for significant markets
    .filter(pl.col("total_vol") > 50000)
)

# 2. Join with Market Questions
whale_report = (
    whale_stats
    .join(df_markets.select(["market_id", "question"]), on="market_id")
    .sort("whale_dominance", descending=True)
    .head(20)
)

print("Markets with Highest Whale Dominance:")
print(whale_report.select(["question", "whale_dominance", "total_vol", "unique_traders"]))

In [ ]:
top_whale_trades = (
    trades_df
    .with_columns((pl.col("price") * pl.col("size")).alias("usd_value"))
    .join(whale_stats.select("market_id"), on="market_id")
    .sort("usd_value", descending=True)
    .group_by("market_id")
    .first() 
    .select(["market_id", "timestamp", "usd_value", "side"])
)

# 3. Join with Market Names 
top_whale_trades = top_whale_trades.join(df_markets.select(["market_id", "question"]), on="market_id")

# 4. Prepare for Join
top_whale_trades = top_whale_trades.with_columns(
    pl.col("timestamp").dt.truncate("1d").cast(pl.Date).alias("date")
)

# convert eda to Polars and ensure the date formats match
eda_pl = pl.from_pandas(eda.reset_index())
eda_pl = eda_pl.with_columns(pl.col("time").dt.cast(pl.Date).alias("date"))

# 5. more discovery, link this to the 2D binning (MVRV and in/outflow)
discovery_table = (
    top_whale_trades
    .join(eda_pl.select(["date", "mvrv_q5", "netflow_q5", "PriceUSD"]), on="date")
    .select(["question", "timestamp", "usd_value", "mvrv_q5", "netflow_q5", "PriceUSD"])
)

print("Discovery: Cross-Referencing Polymarket Big Traders with BTC 2D Binning")
print(discovery_table)